In [1]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("data/global_ads_performance_dataset.csv")

funnel = df.groupby('platform').agg(
    total_clicks=('clicks','sum'),
    total_conversions=('conversions','sum'),
    total_revenue=('revenue','sum'),
    total_spend=('ad_spend','sum'),
    total_impressions=('impressions','sum')
)
funnel['CTR'] = funnel['total_clicks'] / funnel['total_impressions']
funnel['CPC'] = funnel['total_spend'] / funnel['total_clicks']
funnel['CVR'] = funnel['total_conversions'] / funnel['total_clicks']
funnel['AOV'] = funnel['total_revenue'] / funnel['total_conversions']
funnel['ROAS_reconstructed'] = funnel['CVR'] * funnel['AOV'] / funnel['CPC']
funnel['ROAS_actual'] = funnel['total_revenue'] / funnel['total_spend']

print("=== FUNNEL METRICS PER PLATFORM ===")
print(funnel[['CTR','CPC','CVR','AOV','ROAS_actual','ROAS_reconstructed']].round(4))

def decompose(platform_a, platform_b):
    a, b = funnel.loc[platform_a], funnel.loc[platform_b]
    total_log_gap = np.log(a['ROAS_actual'] / b['ROAS_actual'])
    cvr_contrib = np.log(a['CVR'] / b['CVR'])
    aov_contrib = np.log(a['AOV'] / b['AOV'])
    cpc_contrib = -np.log(a['CPC'] / b['CPC'])
    total_check = cvr_contrib + aov_contrib + cpc_contrib

    print(f"\n=== DECOMPOSITION: {platform_a} vs {platform_b} ===")
    print(f"Total ROAS gap (log): {total_log_gap:.3f} (check: {total_check:.3f})")
    print(f"  CVR contribution:  {cvr_contrib:+.3f}  ({cvr_contrib/total_log_gap*100:.0f}% của gap)")
    print(f"  AOV contribution:  {aov_contrib:+.3f}  ({aov_contrib/total_log_gap*100:.0f}% của gap)")
    print(f"  CPC contribution:  {cpc_contrib:+.3f}  ({cpc_contrib/total_log_gap*100:.0f}% của gap)")

    return {
        "total_gap_pct": round((np.exp(total_log_gap)-1)*100,1),
        "cvr_contribution_pct": round(cvr_contrib/total_log_gap*100,1),
        "aov_contribution_pct": round(aov_contrib/total_log_gap*100,1),
        "cpc_contribution_pct": round(cpc_contrib/total_log_gap*100,1),
        "raw_values": {
            platform_a: {k: round(funnel.loc[platform_a,k],4) for k in ['CTR','CPC','CVR','AOV']},
            platform_b: {k: round(funnel.loc[platform_b,k],4) for k in ['CTR','CPC','CVR','AOV']}
        }
    }

decomp_tiktok_vs_google = decompose('TikTok Ads', 'Google Ads')
decomp_meta_vs_google = decompose('Meta Ads', 'Google Ads')

output = {
    "funnel_metrics": funnel[['CTR','CPC','CVR','AOV','ROAS_actual']].round(4).to_dict('index'),
    "decomposition_tiktok_vs_google": decomp_tiktok_vs_google,
    "decomposition_meta_vs_google": decomp_meta_vs_google
}
with open("dashboard_data/step7_funnel_decomposition.json", "w") as f:
    json.dump(output, f, indent=2)
print("\n✅ Saved: dashboard_data/step7_funnel_decomposition.json")

=== FUNNEL METRICS PER PLATFORM ===
               CTR     CPC     CVR       AOV  ROAS_actual  ROAS_reconstructed
platform                                                                     
Google Ads  0.0398  2.1624  0.0446  168.0708       3.4703              3.4703
Meta Ads    0.0248  1.3190  0.0459  162.7862       5.6627              5.6627
TikTok Ads  0.0553  1.0206  0.0471  165.1548       7.6217              7.6217

=== DECOMPOSITION: TikTok Ads vs Google Ads ===
Total ROAS gap (log): 0.787 (check: 0.787)
  CVR contribution:  +0.053  (7% của gap)
  AOV contribution:  -0.018  (-2% của gap)
  CPC contribution:  +0.751  (95% của gap)

=== DECOMPOSITION: Meta Ads vs Google Ads ===
Total ROAS gap (log): 0.490 (check: 0.490)
  CVR contribution:  +0.027  (6% của gap)
  AOV contribution:  -0.032  (-7% của gap)
  CPC contribution:  +0.494  (101% của gap)

✅ Saved: dashboard_data/step7_funnel_decomposition.json
